# Dashboard BI — LattesHub

**Artefato equivalente ao Power BI** (issue #34).

Consome o CSV analítico gerado pelo endpoint `/api/v1/exportacoes/producoes.csv` e exibe:

| KPI / Gráfico | Descrição |
|---|---|
| KPIs de resumo | Total de produções, pesquisadores, média/pesquisador |
| Produções ao longo do tempo | Bar chart por ano |
| Por tipo de produção | Horizontal bar |
| Top áreas de pesquisa | Horizontal bar (top 10) |
| Distribuição Qualis | Stacked bar por estrato |
| Top instituições | Horizontal bar (top 10) |

**Segmentações disponíveis** (células de configuração abaixo):
- `FILTRO_ANO_INICIO` / `FILTRO_ANO_FIM`
- `FILTRO_TIPO` — `'Artigo'`, `'Evento'`, `'Livro'`, `'Capitulo'`, etc.
- `FILTRO_PESQUISADOR` — nome parcial ou ID
- `FILTRO_AREA` — nome parcial da grande área
- `FILTRO_INSTITUICAO` — nome parcial da instituição

Deixe como `None` para incluir tudo.

In [ ]:
# ── Configuração ──────────────────────────────────────────────────────────────
BASE_URL = "http://localhost:8000/api/v1"  # ajuste para seu ambiente
CSV_LOCAL = None  # caminho para CSV já baixado, ex: 'producoes.csv'. None = busca da API

# Segmentações (None = sem filtro)
FILTRO_ANO_INICIO  = None   # ex: 2018
FILTRO_ANO_FIM     = None   # ex: 2024
FILTRO_TIPO        = None   # ex: 'Artigo'
FILTRO_PESQUISADOR = None   # ex: 'Silva'
FILTRO_AREA        = None   # ex: 'Ciências Exatas'
FILTRO_INSTITUICAO = None   # ex: 'UFBA'

In [ ]:
import io
import warnings
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'bold',
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'font.family': 'DejaVu Sans',
})
TEAL   = '#2A9D8F'
SLATE  = '#334155'
LIGHT  = '#F1F5F9'
ACCENT = '#E76F51'

In [ ]:
# ── Carregamento de dados ─────────────────────────────────────────────────────
if CSV_LOCAL:
    df_raw = pd.read_csv(CSV_LOCAL, dtype=str)
    print(f'CSV local carregado: {len(df_raw):,} linhas')
else:
    params = {}
    if FILTRO_ANO_INICIO:  params['ano_inicio'] = FILTRO_ANO_INICIO
    if FILTRO_ANO_FIM:     params['ano_fim']    = FILTRO_ANO_FIM
    if FILTRO_TIPO:        params['tipo_producao'] = FILTRO_TIPO
    url = f'{BASE_URL}/exportacoes/producoes.csv'
    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    df_raw = pd.read_csv(io.StringIO(resp.text), dtype=str)
    print(f'CSV da API carregado: {len(df_raw):,} linhas  |  {url}')

# Tipagem
df_raw['ano'] = pd.to_numeric(df_raw['ano'], errors='coerce')
df = df_raw.copy()

In [ ]:
# ── Segmentações locais (aplicadas após carga) ────────────────────────────────
if FILTRO_PESQUISADOR:
    df = df[df['pesquisador_nome'].str.contains(FILTRO_PESQUISADOR, case=False, na=False)]
if FILTRO_AREA:
    df = df[df['areas'].str.contains(FILTRO_AREA, case=False, na=False)]
if FILTRO_INSTITUICAO:
    df = df[df['instituicao_nome'].str.contains(FILTRO_INSTITUICAO, case=False, na=False)]

print(f'Linhas após filtros: {len(df):,}')

## KPIs de Resumo

In [ ]:
total_producoes    = df['producao_id'].nunique()
total_pesquisadores = df['pesquisador_id'].nunique()
media_por_pesq     = total_producoes / total_pesquisadores if total_pesquisadores else 0
ano_min = int(df['ano'].min()) if df['ano'].notna().any() else '—'
ano_max = int(df['ano'].max()) if df['ano'].notna().any() else '—'

fig, axes = plt.subplots(1, 4, figsize=(14, 2.5))
kpis = [
    ('Total de Produções',       f'{total_producoes:,}',            TEAL),
    ('Pesquisadores',             f'{total_pesquisadores:,}',        SLATE),
    ('Média / Pesquisador',       f'{media_por_pesq:.1f}',           TEAL),
    ('Período coberto',           f'{ano_min} – {ano_max}',          SLATE),
]
for ax, (label, value, color) in zip(axes, kpis):
    ax.set_facecolor(LIGHT)
    ax.text(0.5, 0.62, value, ha='center', va='center', fontsize=22, fontweight='bold', color=color, transform=ax.transAxes)
    ax.text(0.5, 0.22, label, ha='center', va='center', fontsize=9, color='#64748B', transform=ax.transAxes)
    ax.set_axis_off()
fig.suptitle('Indicadores Gerais', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Produções ao Longo do Tempo

In [ ]:
por_ano = (
    df.dropna(subset=['ano'])
      .groupby('ano')['producao_id'].nunique()
      .sort_index()
)

fig, ax = plt.subplots(figsize=(14, 4))
bars = ax.bar(por_ano.index, por_ano.values, color=TEAL, width=0.7, zorder=3)
ax.set_title('Produções ao Longo do Tempo')
ax.set_xlabel('Ano')
ax.set_ylabel('Nº de Produções')
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.grid(axis='y', color='#E2E8F0', zorder=0)
ax.bar_label(bars, fmt='%d', fontsize=7, padding=2)
plt.xticks(por_ano.index, [str(int(a)) for a in por_ano.index], rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

## Por Tipo de Produção

In [ ]:
por_tipo = (
    df.groupby('tipo_producao')['producao_id'].nunique()
      .sort_values()
)

fig, ax = plt.subplots(figsize=(9, max(3, len(por_tipo) * 0.55)))
bars = ax.barh(por_tipo.index, por_tipo.values, color=SLATE, height=0.6)
ax.set_title('Produções por Tipo')
ax.set_xlabel('Nº de Produções')
ax.bar_label(bars, fmt='%d', padding=4, fontsize=9)
ax.set_xlim(0, por_tipo.max() * 1.15)
ax.grid(axis='x', color='#E2E8F0', zorder=0)
plt.tight_layout()
plt.show()

## Top Áreas de Pesquisa

In [ ]:
areas_series = (
    df[df['areas'].notna() & (df['areas'] != '')]
      .assign(grande_area=lambda d: d['areas'].str.split(' > ').str[0].str.strip())
      .groupby('grande_area')['producao_id'].nunique()
      .sort_values(ascending=True)
      .tail(10)
)

fig, ax = plt.subplots(figsize=(9, max(3, len(areas_series) * 0.55)))
colors = [TEAL if i == len(areas_series) - 1 else SLATE for i in range(len(areas_series))]
bars = ax.barh(areas_series.index, areas_series.values, color=colors, height=0.6)
ax.set_title('Top 10 Áreas de Pesquisa (por Nº de Produções)')
ax.set_xlabel('Nº de Produções')
ax.bar_label(bars, fmt='%d', padding=4, fontsize=9)
ax.set_xlim(0, areas_series.max() * 1.15)
ax.grid(axis='x', color='#E2E8F0', zorder=0)
plt.tight_layout()
plt.show()

## Distribuição Qualis

In [ ]:
ordem_qualis = ['A1', 'A2', 'A3', 'A4', 'B1', 'B2', 'B3', 'B4', 'C', 'Sem Qualis']

qualis_df = df.copy()
qualis_df['qualis_estrato'] = qualis_df['qualis_estrato'].fillna('Sem Qualis')
por_qualis = (
    qualis_df.groupby('qualis_estrato')['producao_id'].nunique()
             .reindex([s for s in ordem_qualis if s in qualis_df['qualis_estrato'].unique()])
             .dropna()
)

palette = {
    'A1': '#0F5132', 'A2': '#198754', 'A3': '#20C997', 'A4': '#75B798',
    'B1': '#084298', 'B2': '#0D6EFD', 'B3': '#6EA8FE', 'B4': '#9EC5FE',
    'C':  '#9E2A2B', 'Sem Qualis': '#CBD5E1',
}
bar_colors = [palette.get(s, SLATE) for s in por_qualis.index]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(por_qualis.index, por_qualis.values, color=bar_colors, width=0.6, zorder=3)
ax.set_title('Distribuição por Estrato Qualis')
ax.set_ylabel('Nº de Produções')
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.grid(axis='y', color='#E2E8F0', zorder=0)
ax.bar_label(bars, fmt='%d', fontsize=8, padding=2)
plt.tight_layout()
plt.show()

## Top Instituições

In [ ]:
por_inst = (
    df[df['instituicao_nome'].notna()]
      .groupby('instituicao_nome')['producao_id'].nunique()
      .sort_values(ascending=True)
      .tail(10)
)

fig, ax = plt.subplots(figsize=(10, max(3, len(por_inst) * 0.55)))
bars = ax.barh(por_inst.index, por_inst.values, color=TEAL, height=0.6)
ax.set_title('Top 10 Instituições (por Nº de Produções)')
ax.set_xlabel('Nº de Produções')
ax.bar_label(bars, fmt='%d', padding=4, fontsize=9)
ax.set_xlim(0, por_inst.max() * 1.15)
ax.grid(axis='x', color='#E2E8F0', zorder=0)
plt.tight_layout()
plt.show()

## Evolução por Tipo (séries temporais)

In [ ]:
top_tipos = df['tipo_producao'].value_counts().head(4).index.tolist()
pivot = (
    df[df['tipo_producao'].isin(top_tipos) & df['ano'].notna()]
      .groupby(['ano', 'tipo_producao'])['producao_id'].nunique()
      .unstack(fill_value=0)
      .sort_index()
)

colors_line = [TEAL, SLATE, ACCENT, '#8B5CF6']
fig, ax = plt.subplots(figsize=(13, 4))
for col, color in zip(pivot.columns, colors_line):
    ax.plot(pivot.index, pivot[col], marker='o', markersize=4, linewidth=2, label=col, color=color)
ax.set_title('Evolução por Tipo de Produção (Top 4)')
ax.set_xlabel('Ano')
ax.set_ylabel('Nº de Produções')
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.grid(color='#E2E8F0')
ax.legend(loc='upper left', fontsize=8)
plt.xticks(pivot.index, [str(int(a)) for a in pivot.index], rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

## Tabela resumo — segmentações ativas

In [ ]:
summary = pd.DataFrame({
    'Métrica': [
        'Total de produções (filtradas)',
        'Pesquisadores distintos',
        'Instituições distintas',
        'Áreas distintas (grande área)',
        'Tipos de produção distintos',
        'Anos cobertos',
    ],
    'Valor': [
        f'{total_producoes:,}',
        f'{total_pesquisadores:,}',
        f"{df['instituicao_nome'].nunique():,}",
        f"{df[df['areas'].notna()].assign(ga=lambda d: d['areas'].str.split(' > ').str[0])['ga'].nunique():,}",
        f"{df['tipo_producao'].nunique():,}",
        f'{ano_min} – {ano_max}',
    ]
})
summary.style.set_caption('Resumo — LattesHub BI').hide(axis='index')